# Dhara — Topic 5, zero-shot bi-encoder, LARGE checkpoint

Same pipeline as `topic5_zeroshot_biencoder.ipynb`, checkpoint swapped from
`multilingual-e5-base` (768-dim) to **`multilingual-e5-large`** (1024-dim).

**Why this run exists.** The base zero-shot index was measured against 7
hand-verified questions (`data/interim/zeroshot_probe_pairs.jsonl`) and split
cleanly by source language: both Bengali-source answers were found near the
top, all five English-source answers (CrPC, the Negotiable Instruments Act, the
State Acquisition and Tenancy Act — all pre-1950 English legal prose) were
buried past rank 8,000 of 39,484. Overall Recall@50 was 1/7. This run checks
whether a larger checkpoint closes that specific gap, or whether the gap is a
ceiling zero-shot cannot cross regardless of model size — which is the more
likely outcome and the reason Topic 5 is fine-tuned at all, not a reason to
keep swapping checkpoints if this one also fails.

**Before running:** Runtime → Change runtime type → **T4 GPU**. This checkpoint
is roughly 3.5x the parameters of base — indexing will take longer (rough
estimate 20–40 min for 39,484 chunks) and `batch_size` is lowered to 32 to fit
T4 memory.

**What you need on hand:** the same `models/corpus_v1.jsonl.gz` you already
have from the base run.

**What you get back:** the same probe numbers printed inline (no waiting on a
round trip to compare), plus `index_v1_large.zip` if you want to keep the index.

In [ ]:
!pip install -q sentence-transformers

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — go to Runtime > Change runtime type > T4 GPU, then re-run")

## 1. Upload the corpus

Upload `corpus_v1.jsonl.gz` from `models/` in the repo. If the upload widget
times out or the file is bigger than expected (it should be ~12MB), mount Drive
instead — the commented alternative is below.

In [ ]:
from google.colab import files
uploaded = files.upload()  # pick corpus_v1.jsonl.gz
CORPUS_GZ = next(iter(uploaded))
print("got:", CORPUS_GZ)

# --- alternative: Google Drive, if the upload widget is unreliable on your connection ---
# from google.colab import drive
# drive.mount('/content/drive')
# CORPUS_GZ = "/content/drive/MyDrive/corpus_v1.jsonl.gz"  # adjust to where you put it

In [ ]:
import gzip, shutil, pathlib

CORPUS_PATH = pathlib.Path("corpus_v1.jsonl")
with gzip.open(CORPUS_GZ, "rb") as f_in, CORPUS_PATH.open("wb") as f_out:
    shutil.copyfileobj(f_in, f_out)
print(f"{CORPUS_PATH.stat().st_size / 1e6:.1f} MB decompressed")

## 2. The code being run

Inlined rather than cloned from the repo, so this notebook runs the same way
whether or not tonight's fixes have been pushed yet. It is a direct copy of
`src/dhara/retrievers/biencoder.py` plus the minimal pieces of `schema.py` and
`normalize.py` it depends on — **if you change the checkpoint or the prefix
logic in the real repo, update it here too**, or the zero-shot run and the
fine-tuned run stop being comparable.

In [ ]:
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass
from typing import Iterator, Optional

import numpy as np
from sentence_transformers import SentenceTransformer

# ---- normalize.light(), copied verbatim from src/dhara/normalize.py ----
ZWNJ, ZWJ = "\u200c", "\u200d"
DANDA, DANDA_VARIANTS = "\u0964", "\u09f7"
_WS = re.compile(r"[ \t\u00a0]+")
_BLANKS = re.compile(r"\n{3,}")

def light(text: str) -> str:
    """For transformer inputs. Fix encoding inconsistencies only — never
    aggressive-normalize text going into a transformer, that is a measurable
    performance loss (see DECISIONS.md / CLAUDE.md)."""
    text = unicodedata.normalize("NFC", text)
    text = text.replace(ZWNJ, "")
    text = text.replace(DANDA_VARIANTS, DANDA)
    text = text.replace("\xa0", " ")
    text = _WS.sub(" ", text)
    text = _BLANKS.sub("\n\n", text)
    return text.strip()

# ---- schema.Chunk, only the fields the bi-encoder touches ----
@dataclass
class Chunk:
    chunk_id: str
    provision_id: str
    act_id: str
    act_title_bn: str
    act_title_en: str
    provision_kind: str
    provision_no_bn: str
    provision_title_bn: Optional[str]
    text_bn: str
    domain: str
    language: str

def read_jsonl(path) -> Iterator[Chunk]:
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            if line.strip():
                row = json.loads(line)
                yield Chunk(**{k: row.get(k) for k in Chunk.__dataclass_fields__})

# ---- retrievers/biencoder.py, copied verbatim ----
def prefixes_for(checkpoint: str) -> tuple[str, str]:
    """E5 checkpoints need "query: " / "passage: " prefixes or the baseline is
    silently crippled. Derived from the checkpoint name so the zero-shot and
    fine-tuned runs cannot drift apart."""
    name = checkpoint.lower()
    if "e5" in name:
        return "query: ", "passage: "
    if "bge" in name and "m3" not in name:
        return "Represent this sentence for searching relevant passages: ", ""
    return "", ""

class BiEncoderRetriever:
    name = "biencoder"

    def __init__(self, checkpoint="intfloat/multilingual-e5-base",
                 max_seq_length=256, batch_size=64, device=None, use_title=True):
        self.checkpoint = checkpoint
        self.max_seq_length = max_seq_length
        self.batch_size = batch_size
        self.device = device
        self.use_title = use_title
        self.query_prefix, self.passage_prefix = prefixes_for(checkpoint)
        self.ids = []
        self.embeddings = None
        self._model = None

    @property
    def model(self):
        if self._model is None:
            self._model = SentenceTransformer(self.checkpoint, device=self.device)
            self._model.max_seq_length = self.max_seq_length
        return self._model

    def _document(self, chunk: Chunk) -> str:
        parts = []
        if self.use_title and chunk.provision_title_bn:
            parts.append(chunk.provision_title_bn)
        parts.append(chunk.text_bn)
        return self.passage_prefix + light(" ".join(parts))

    def index(self, chunks) -> None:
        self.ids = [c.chunk_id for c in chunks]
        texts = [self._document(c) for c in chunks]
        self.embeddings = self.model.encode(
            texts, batch_size=self.batch_size, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=True,
        )

    def search(self, query: str, k: int = 10):
        vector = self.model.encode(
            [self.query_prefix + light(query)], convert_to_numpy=True,
            normalize_embeddings=True,
        )[0]
        scores = self.embeddings @ vector
        k = min(k, len(scores))
        top = np.argpartition(scores, -k)[-k:]
        top = top[np.argsort(scores[top])[::-1]]
        return [(self.ids[i], float(scores[i])) for i in top]

    def save(self, directory: pathlib.Path) -> None:
        directory.mkdir(parents=True, exist_ok=True)
        np.save(directory / "embeddings.npy", self.embeddings)
        (directory / "chunk_ids.json").write_text(json.dumps(self.ids), encoding="utf-8")

print("code loaded")

## 3. Load the corpus and build the index

~39,500 chunks on a T4 with batch_size=64 should take roughly 5–15 minutes.
This is the only slow cell in the notebook.

In [ ]:
chunks = list(read_jsonl(CORPUS_PATH))
by_id = {c.chunk_id: c for c in chunks}
print(f"{len(chunks)} chunks loaded")

CHECKPOINT = "intfloat/multilingual-e5-large"
retriever = BiEncoderRetriever(checkpoint=CHECKPOINT, batch_size=32)  # large, not base
print(f"checkpoint: {retriever.checkpoint}")
print(f"prefixes:   query={retriever.query_prefix!r}  passage={retriever.passage_prefix!r}")

retriever.index(chunks)
print(f"embeddings shape: {retriever.embeddings.shape}")

## 4. Sanity check — does it actually cross the language gap?

This is the specific failure BM25 has: a Bangla question cannot lexically match
an English provision. Three questions below have their true answer in an
English-only Act. If the top hits land in the right Act, the zero-shot control
is doing its job; if not, something is wrong before you trust anything else out
of this notebook (wrong checkpoint, missing prefixes, bad normalization).

In [ ]:
checks = [
    ("কোনো মেয়ের বাবা জীবিত থাকা অবস্থায় মারা গেলে মেয়ের ছেলে-মেয়ে কি মায়ের অংশ পাবে?",
     "Muslim Family Laws Ordinance"),
    ("জমি রেজিস্ট্রি করতে কী কাগজপত্র লাগবে?",
     "Registration Act"),
    ("পুলিশ ওয়ারেন্ট ছাড়া কাউকে গ্রেপ্তার করা যায় কিনা?",
     "Code of Criminal Procedure"),
]

for question, expect in checks:
    print(f"\nQ: {question}\n   (expect an act containing {expect!r} in its results)")
    for chunk_id, score in retriever.search(question, k=5):
        c = by_id[chunk_id]
        title = c.act_title_bn or c.act_title_en
        flag = " <-- MATCH" if expect.lower() in (c.act_title_en or "").lower() else ""
        print(f"   {score:.3f}  {title[:46]:48s} ধারা {c.provision_no_bn}{flag}")

## 4b. The actual question: does LARGE close the language gap?

The same 7 hand-verified pairs used to score the base checkpoint, run here so
the comparison needs no round trip. `overall`, `english`, `bengali` columns
mirror `scripts/12_zeroshot_probe.py` — paste this cell's printed table back
for a direct before/after.

In [ ]:
PROBES = [{"qid": "probe_01", "question_bn": "স্বামী তালাক দিলে চেয়ারম্যানকে নোটিশ দিতে হয় কিনা?", "chunk_id": "family_305_s7", "act": "Muslim Family Laws Ordinance, 1961", "language": "english", "verified_by": "content read, not title-only"}, {"qid": "probe_02", "question_bn": "থানায় কীভাবে এজাহার (এফআইআর) করতে হয়?", "chunk_id": "criminal_procedure_75_s154", "act": "The Code of Criminal Procedure, 1898", "language": "english", "verified_by": "content read, not title-only"}, {"qid": "probe_03", "question_bn": "অজামিনযোগ্য মামলায় জামিন পাওয়ার নিয়ম কী?", "chunk_id": "criminal_procedure_75_s497", "act": "The Code of Criminal Procedure, 1898", "language": "english", "verified_by": "content read, not title-only"}, {"qid": "probe_04", "question_bn": "চেক ডিজঅনার হলে কী করব?", "chunk_id": "money_recovery_46_s92", "act": "The Negotiable Instruments Act, 1881", "language": "english", "verified_by": "content read, not title-only"}, {"qid": "probe_05", "question_bn": "জমি বিক্রির ক্ষেত্রে প্রতিবেশীর অগ্রক্রয়ের অধিকার আছে কিনা?", "chunk_id": "land_241_s96_p1", "act": "The State Acquisition and Tenancy Act, 1950", "language": "english", "verified_by": "content read, not title-only"}, {"qid": "probe_06", "question_bn": "তথ্য অধিকার আইনে চাওয়া তথ্য না পেলে কী করব?", "chunk_id": "consumer_1011_s24", "act": "তথ্য অধিকার আইন, ২০০৯", "language": "bengali", "verified_by": "content read, not title-only"}, {"qid": "probe_07", "question_bn": "বিয়েতে যৌতুক দাবি করলে শাস্তি কী?", "chunk_id": "women_children_1256_s3", "act": "যৌতুক নিরোধ আইন, ২০১৮", "language": "bengali", "verified_by": "content read, not title-only"}]

CUTOFFS = (5, 10, 20, 50, 100)

def rank_of(question, target, k):
    for i, (cid, _s) in enumerate(retriever.search(question, k=k), 1):
        if cid == target:
            return i
    return None

ranks = {}
header = "{:10s} {:8s} {:>8s}  question".format("qid", "lang", "rank")
print(header)
for p in PROBES:
    r_ = rank_of(p['question_bn'], p['chunk_id'], len(chunks))
    ranks[p['qid']] = r_
    row = "{:10s} {:8s} {:>8s}  {}".format(p["qid"], p["language"], str(r_), p["question_bn"][:50])
    print(row)

from collections import defaultdict
by_lang = defaultdict(list)
for p in PROBES:
    by_lang[p['language']].append(ranks[p['qid']])
all_ranks = list(ranks.values())

print()
print("{:10s} {:>10s} {:>10s} {:>10s}".format("cutoff", "overall", "english", "bengali"))
for k in CUTOFFS:
    def hit(rs):
        rs = [r for r in rs if r is not None]
        return "{}/{}".format(sum(1 for r in rs if r <= k), len(rs)) if rs else "-"
    line = "Recall@{:<4d}{:>10s}{:>10s}{:>10s}".format(
        k, hit(all_ranks), hit(by_lang.get('english', [])), hit(by_lang.get('bengali', [])))
    print(line)

print()
print("compare against the base checkpoint result:")
print("  base:  overall 1/7  english 0/5  bengali 1/2  (Recall@50)")


## 5. Save the index and download it

Produces `index_v1_large.zip`. Unzip it into `models/index_v1_large/` in the repo —
`BiEncoderRetriever.load()` reads `embeddings.npy` + `chunk_ids.json` from there.

**This save is the artifact that matters most from this notebook.** It is the
locked zero-shot baseline — do not regenerate it once fine-tuning starts, or
the "transformers work" vs "our fine-tuning works" comparison loses its anchor.

In [ ]:
OUT = pathlib.Path("index_v1_large")
retriever.save(OUT)

# record exactly what produced this index, so it's reproducible without re-reading this notebook
(OUT / "manifest.json").write_text(json.dumps({
    "checkpoint": retriever.checkpoint,
    "max_seq_length": retriever.max_seq_length,
    "use_title": retriever.use_title,
    "query_prefix": retriever.query_prefix,
    "passage_prefix": retriever.passage_prefix,
    "n_chunks": len(chunks),
    "corpus_file": "corpus_v1.jsonl",
    "fine_tuned": False,
}, indent=2, ensure_ascii=False), encoding="utf-8")

import shutil
shutil.make_archive("index_v1_large", "zip", OUT)
print(f"{pathlib.Path('index_v1_large.zip').stat().st_size / 1e6:.1f} MB")

from google.colab import files
files.download("index_v1_large.zip")

## 6. (Optional, but do this) Run it against a few gold questions now

You don't need the full evaluation script yet — just paste a handful of real
questions from `data/annotation/*.csv` here and eyeball the top-5 against the
advocate's `lawyer_answer` column, or against your own judgement. This is the
fastest way to catch "wrong checkpoint" or "prefixes backwards" before you
download 300MB+ of embeddings and find out later.

In [ ]:
# Paste real questions from your annotation sheets here and re-run this cell as needed.
my_questions = [
    "একটি কো-অপারেটিভ সংস্থার কাছে আমার অনেক টাকা আটকে আছে। কী করব?",
]
for q in my_questions:
    print(f"\nQ: {q}")
    for chunk_id, score in retriever.search(q, k=5):
        c = by_id[chunk_id]
        print(f"   {score:.3f}  {(c.act_title_bn or c.act_title_en)[:46]:48s} ধারা {c.provision_no_bn}")